# Forma HOOD Validation Notebook

Runs on **Kaggle free T4 GPU** (GPU accelerator must be enabled in Settings).

Validates the ContourCraft / HOOD simulation backend against the CPU XPBD solver by comparing `clearance_map` values per body region. Each region delta must be ≤ 2 mm.

## What this notebook does
1. Clone the Forma repo
2. Install ContourCraft and all its dependencies
3. Download pretrained ContourCraft weights
4. Run the **CPU solver** on `makehuman_male_M.ply` + `tshirt_size_M.json`
5. Run the **HOOD solver** on the same inputs
6. Compare `clearance_map` per region — flag any delta > 2 mm as FAIL

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())
assert result.returncode == 0, 'GPU not available — enable GPU accelerator in Kaggle Settings'

## Step 1 — Clone Forma repo

In [ ]:
# Forma repo on GitHub
FORMA_REPO = 'https://github.com/cmpotter04-sys/forma.git'
FORMA_BRANCH = 'main'
FORMA_DIR = '/kaggle/working/Forma'

!git clone --depth 1 --branch {FORMA_BRANCH} {FORMA_REPO} {FORMA_DIR}
print('Forma cloned to', FORMA_DIR)

## Step 2 — Install ContourCraft and dependencies

In [ ]:
# ContourCraft repo
CC_DIR = '/kaggle/working/ContourCraft'
!git clone --depth 1 https://github.com/Dolorousrtur/ContourCraft.git {CC_DIR}

In [ ]:
# Detect torch + CUDA version to choose the correct pyg wheel URL
import torch
torch_ver = torch.__version__.split('+')[0]          # e.g. '2.5.0'
cuda_ver  = torch.version.cuda.replace('.', '')      # e.g. '124'
print(f'torch={torch_ver}  cuda={cuda_ver}')

PYG_WHL = f'https://data.pyg.org/whl/torch-{torch_ver}+cu{cuda_ver}.html'

In [ ]:
# Install torch-geometric + sparse dependencies
!pip install -q torch-geometric==2.4.0
!pip install -q pyg_library torch_scatter torch_sparse torch_cluster torch_spline_conv -f {PYG_WHL}

In [ ]:
# Install pytorch3d (takes ~5 min to build wheel from source)
!pip install -q "git+https://github.com/facebookresearch/pytorch3d.git@stable"

In [ ]:
# Install CCCollision (ContourCraft's CUDA collision solver)
!git clone -q https://github.com/NVIDIA/cuda-samples.git /tmp/cuda-samples
import os; os.environ['CUDA_SAMPLES_INC'] = '/tmp/cuda-samples/Common'
!git clone -q https://github.com/Dolorousrtur/CCCollisions.git /tmp/CCCollisions
!pip install -q /tmp/CCCollisions

In [ ]:
# Remaining ContourCraft dependencies
!pip install -q omegaconf einops networkx munch trimesh scipy scikit-learn loguru

In [ ]:
# Forma dependencies (CPU solver + pattern loading)
!pip install -q trimesh scipy numpy ezdxf shapely

## Step 3 — Download ContourCraft pretrained weights

Download `contourcraft.pth` from the [Google Drive link](https://drive.google.com/file/d/1NfxAeaC2va8TWMjiO_gbAcVPnZ8BYFPD/view) and upload it to Kaggle as a dataset, or use `gdown` if the file is publicly accessible.

In [ ]:
import os

# Option A: gdown (works if Google Drive sharing is set to 'Anyone with the link')
GDRIVE_FILE_ID = '1NfxAeaC2va8TWMjiO_gbAcVPnZ8BYFPD'
CHECKPOINT_DIR = f'{CC_DIR}/trained_models'
CHECKPOINT_PATH = f'{CHECKPOINT_DIR}/contourcraft.pth'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(CHECKPOINT_PATH):
    !pip install -q gdown
    !gdown {GDRIVE_FILE_ID} -O {CHECKPOINT_PATH}

assert os.path.exists(CHECKPOINT_PATH), (
    f'Checkpoint not found at {CHECKPOINT_PATH}. '
    'Download contourcraft.pth manually and place it at this path.'
)
print(f'Checkpoint found: {CHECKPOINT_PATH}  ({os.path.getsize(CHECKPOINT_PATH) // 1024 / 1024:.1f} MB)')

## Step 4 — Configure paths and run CPU solver

In [ ]:
import sys

FORMA_SRC = f'{FORMA_DIR}/src'
if FORMA_SRC not in sys.path:
    sys.path.insert(0, FORMA_SRC)

# Paths to test assets
BODY_MESH     = f'{FORMA_DIR}/data/bodies/makehuman_male_M.ply'
PATTERN       = f'{FORMA_DIR}/data/patterns/tshirt_size_M.json'
SEAM_MANIFEST = f'{FORMA_DIR}/seam_manifests/tshirt_size_M_manifest.json'
FABRIC_ID     = 'cotton_jersey_default'

# Verify test assets exist
for p in [BODY_MESH, PATTERN, SEAM_MANIFEST]:
    assert os.path.exists(p), f'Missing test asset: {p}'
print('All test assets found.')

In [ ]:
import time
from pipeline import run_fit_check

print('Running CPU solver...')
t0 = time.perf_counter()
cpu_verdict = run_fit_check(
    body_mesh_path=BODY_MESH,
    pattern_path=PATTERN,
    seam_manifest_path=SEAM_MANIFEST,
    fabric_id=FABRIC_ID,
    backend='cpu',
)
cpu_ms = int((time.perf_counter() - t0) * 1000)
print(f'CPU solver done in {cpu_ms} ms')

cpu_clearance = {r['region']: r['delta_mm'] for r in cpu_verdict['strain_map']}
print('\nCPU clearance_map:')
for region, delta in sorted(cpu_clearance.items()):
    print(f'  {region:20s}: {delta:+.1f} mm')

## Step 5 — Run HOOD solver

In [ ]:
import os
os.environ['CONTOURCRAFT_ROOT'] = CC_DIR
os.environ['CONTOURCRAFT_CHECKPOINT'] = CHECKPOINT_PATH

# Set ContourCraft defaults so its internal path resolution works
cc_root_inserted = CC_DIR not in sys.path
if cc_root_inserted:
    sys.path.insert(0, CC_DIR)

from utils.defaults import DEFAULTS
DEFAULTS.project_dir = CC_DIR
DEFAULTS.data_root   = CC_DIR   # aux_data lives under here; we just need path resolution

print('Running HOOD solver...')
t0 = time.perf_counter()
hood_verdict = run_fit_check(
    body_mesh_path=BODY_MESH,
    pattern_path=PATTERN,
    seam_manifest_path=SEAM_MANIFEST,
    fabric_id=FABRIC_ID,
    backend='hood',
)
hood_ms = int((time.perf_counter() - t0) * 1000)
print(f'HOOD solver done in {hood_ms} ms')

hood_clearance = {r['region']: r['delta_mm'] for r in hood_verdict['strain_map']}
print('\nHOOD clearance_map:')
for region, delta in sorted(hood_clearance.items()):
    print(f'  {region:20s}: {delta:+.1f} mm')

## Step 6 — Compare results (pass if all region deltas ≤ 2 mm)

In [ ]:
TOLERANCE_MM = 2.0

print(f'{'Region':<22} {'CPU (mm)':>10} {'HOOD (mm)':>10} {'|diff| (mm)':>12} {'Result':>8}')
print('-' * 66)

all_pass = True
for region in sorted(cpu_clearance.keys()):
    cpu_val  = cpu_clearance[region]
    hood_val = hood_clearance.get(region, float('nan'))
    diff     = abs(cpu_val - hood_val)
    passed   = diff <= TOLERANCE_MM
    status   = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'{region:<22} {cpu_val:>+10.2f} {hood_val:>+10.2f} {diff:>12.2f} {status:>8}')

print('-' * 66)
print()

if all_pass:
    print(f'OVERALL: PASS — all regions within {TOLERANCE_MM} mm tolerance')
else:
    print(f'OVERALL: FAIL — one or more regions exceed {TOLERANCE_MM} mm tolerance')
    raise AssertionError(
        f'HOOD vs CPU clearance delta exceeds {TOLERANCE_MM} mm tolerance on one or more regions.'
    )

## Summary

In [ ]:
print('=== Validation Summary ===')
print(f'CPU solver:  {cpu_ms} ms   fit={cpu_verdict["fit"]}')
print(f'HOOD solver: {hood_ms} ms  fit={hood_verdict["fit"]}')
print(f'Tolerance:   {TOLERANCE_MM} mm per region')
print(f'Result:      {"PASS" if all_pass else "FAIL"}')